In [ ]:
# -*- coding: utf-8 -*-
"""
=================================================================
  스트레스 지수 예측 - SVR Multi-Seed Ensemble
=================================================================

[개발 환경]
  - OS          : Google Colab (Ubuntu 22.04)
  - Python      : 3.10.12
  - pandas      : 2.2.2
  - numpy       : 1.26.4
  - scikit-learn: 1.6.1
  - hyperopt    : 0.2.7

[모델 개요]
  - Base Model  : SVR (RBF kernel) + QuantileTransformer (target) + RobustScaler (features)
  - 앙상블      : 10-Fold × 10-Seed = 100개 모델 평균 + Full-train 블렌딩
  - HP 탐색     : Hyperopt (TPE, 100회)
  - CV MAE      : 약 0.138

[데이터 입출력 경로]
  - 입력 : /data/train.csv, /data/test.csv, /data/sample_submission.csv
  - 출력 : /data/submission.csv, /data/submission_eps0.csv
"""

# ============================================================
# 1. 라이브러리 임포트
# ============================================================
import pandas as pd
import numpy as np
from sklearn.svm import SVR
from sklearn.preprocessing import RobustScaler, QuantileTransformer
from sklearn.compose import TransformedTargetRegressor
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK
import warnings
warnings.filterwarnings('ignore')

# 버전 확인
import sklearn, hyperopt as hopt
print(f"pandas     : {pd.__version__}")
print(f"numpy      : {np.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"hyperopt   : {hopt.__version__}")

# ============================================================
# 2. 전역 설정
# ============================================================
DATA_PATH      = '/data/'   # 대회 규칙: '/data/' 경로 사용
N_SPLITS       = 10         # K-Fold 분할 수
HYPEROPT_EVALS = 100        # 하이퍼파라미터 탐색 횟수
RANDOM_STATE   = 42         # 재현성을 위한 시드

print(f"\n[설정] DATA_PATH={DATA_PATH}, N_SPLITS={N_SPLITS}, "
      f"HYPEROPT_EVALS={HYPEROPT_EVALS}")
print("=" * 65)

# ============================================================
# 3. 데이터 로드
# ============================================================
train  = pd.read_csv(f'{DATA_PATH}train.csv')
test   = pd.read_csv(f'{DATA_PATH}test.csv')
submit = pd.read_csv(f'{DATA_PATH}sample_submission.csv')

print(f"Train shape : {train.shape}")
print(f"Test shape  : {test.shape}")
print(f"Target 통계 : mean={train['stress_score'].mean():.4f}, "
      f"std={train['stress_score'].std():.4f}, "
      f"min={train['stress_score'].min():.4f}, "
      f"max={train['stress_score'].max():.4f}")

# 타깃(y)과 ID 분리
y        = train['stress_score'].copy()
train_ids = train['ID'].copy()
test_ids  = test['ID'].copy()

# 학습/테스트에서 ID, target 컬럼 제거
train = train.drop(['ID', 'stress_score'], axis=1)
test  = test.drop(['ID'], axis=1)

# ============================================================
# 4. 전처리
# ============================================================
print("\n[전처리 시작]")

# --- 4-1. 결측치 처리 ---
# mean_working: 결측 → 0 (근무시간 없음으로 해석)
train['mean_working'] = train['mean_working'].fillna(0)
test['mean_working']  = test['mean_working'].fillna(0)

# 나머지 결측 → 'Unknown' (범주형 컬럼)
train = train.fillna('Unknown')
test  = test.fillna('Unknown')

# --- 4-2. 범주형 변수 인코딩 (순서형 매핑) ---
mapping_dict = {
    'gender':        {"F": 0, "M": 1},
    'activity':      {"light": 0, "moderate": 1, "intense": 2},
    'smoke_status':  {"non-smoker": 0, "ex-smoker": 1, "current-smoker": 2},
    'edu_level':     {'high school diploma': 1, 'bachelors degree': 2,
                      'graduate degree': 3, 'Unknown': 0},
    'sleep_pattern': {'sleep difficulty': 0, 'normal': 1, 'oversleeping': 2}
}

for col, d_map in mapping_dict.items():
    train[col] = train[col].map(d_map)
    test[col]  = test[col].map(d_map)

# --- 4-3. One-Hot Encoding (SVR에 효과적) ---
ohe_targets = [
    ('mh',  'medical_history'),
    ('fmh', 'family_medical_history'),
    ('smo', 'smoke_status')
]

for prefix, col in ohe_targets:
    train_dum = pd.get_dummies(train[col], prefix=prefix, dtype='int')
    test_dum  = pd.get_dummies(test[col],  prefix=prefix, dtype='int')
    # train/test 컬럼 일치시키기
    for c in train_dum.columns:
        if c not in test_dum.columns:
            test_dum[c] = 0
    for c in test_dum.columns:
        if c not in train_dum.columns:
            train_dum[c] = 0
    test_dum = test_dum[train_dum.columns]
    train = pd.concat([train, train_dum], axis=1)
    test  = pd.concat([test,  test_dum],  axis=1)

# 원본 범주형 컬럼 제거
drop_cols = ['medical_history', 'family_medical_history', 'smoke_status']
train = train.drop(drop_cols, axis=1)
test  = test.drop(drop_cols,  axis=1)

print(f"  전처리 완료 → Train: {train.shape}, Test: {test.shape}")

# ============================================================
# 5. 피처 엔지니어링
# ============================================================
print("[피처 엔지니어링]")

for df in [train, test]:
    # BMI (체질량지수)
    df['bmi'] = (df['weight'] / ((df['height'] / 100.0) ** 2)).round(2)

    # 맥압 (Pulse Pressure) = 수축기 - 이완기
    df['pulse_pressure'] = df['systolic_blood_pressure'] - df['diastolic_blood_pressure']

    # 평균동맥압 (MAP) = 이완기 + 맥압/3
    df['map_bp'] = df['diastolic_blood_pressure'] + df['pulse_pressure'] / 3.0

    # 교호작용: 나이 × 활동량
    df['age_activity'] = df['age'] * df['activity']

    # 교호작용: 나이 × BMI
    df['age_bmi'] = df['age'] * df['bmi']

print(f"  피처 엔지니어링 완료 → Train: {train.shape}, Test: {test.shape}")
print(f"  최종 피처: {list(train.columns)}")

# numpy 배열 변환
X      = train.values.astype(np.float64)
X_test = test.values.astype(np.float64)
y_arr  = y.values.astype(np.float64)

# ============================================================
# 6. 헬퍼 함수
# ============================================================
def build_svr_pipeline(params, n_train):
    """최적 파라미터로 SVR 파이프라인을 생성한다.

    Args:
        params  : dict - SVR 하이퍼파라미터 (C, gamma, epsilon)
        n_train : int  - 학습 데이터 샘플 수 (QuantileTransformer용)

    Returns:
        sklearn.pipeline.Pipeline
    """
    return make_pipeline(
        RobustScaler(),
        TransformedTargetRegressor(
            regressor=SVR(
                kernel="rbf",
                C=params["C"],
                gamma=params["gamma"],
                epsilon=params["epsilon"],
                shrinking=True,
                cache_size=500,
                max_iter=-1
            ),
            transformer=QuantileTransformer(
                output_distribution="normal",
                n_quantiles=min(1000, n_train)
            )
        )
    )

# ============================================================
# 7. Hyperopt 하이퍼파라미터 탐색
# ============================================================
def objective(params):
    """SVR 하이퍼파라미터 탐색을 위한 목적함수.

    파이프라인: RobustScaler → SVR(RBF) + QuantileTransformer(target)
    평가지표 : 10-Fold CV MAE
    """
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    fold_maes = []

    for tr_idx, val_idx in kf.split(X):
        pipe = build_svr_pipeline(params, len(y_arr[tr_idx]))
        pipe.fit(X[tr_idx], y_arr[tr_idx])
        pred = pipe.predict(X[val_idx])
        fold_maes.append(mean_absolute_error(y_arr[val_idx], pred))

    return {"loss": float(np.mean(fold_maes)), "status": STATUS_OK}


# 탐색 공간 정의
space = {
    "C":       hp.loguniform("C",       np.log(0.5),  np.log(50)),
    "gamma":   hp.loguniform("gamma",   np.log(0.05), np.log(5)),
    "epsilon": hp.loguniform("epsilon", np.log(1e-6), np.log(0.1)),
}

print(f"\n[Hyperopt] 탐색 시작 ({HYPEROPT_EVALS}회)...")
trials = Trials()
best = fmin(
    fn=objective,
    space=space,
    algo=tpe.suggest,
    max_evals=HYPEROPT_EVALS,
    trials=trials,
    rstate=np.random.default_rng(RANDOM_STATE)
)

# 최적 파라미터 추출
best_params = {
    "C":       float(best["C"]),
    "gamma":   float(best["gamma"]),
    "epsilon": float(best["epsilon"])
}
best_mae = float(min(r["loss"] for r in trials.results))

print(f"\n  최적 CV MAE  : {best_mae:.5f}")
print(f"  최적 파라미터: C={best_params['C']:.4f}, "
      f"gamma={best_params['gamma']:.4f}, "
      f"epsilon={best_params['epsilon']:.6f}")

# ============================================================
# 8. 모델 학습 및 예측
# ============================================================

# --- 8-1. 방법 1: Full Train 학습 ---
print("\n[방법 1] Full Train 학습")
pipe_full = build_svr_pipeline(best_params, len(y_arr))
pipe_full.fit(X, y_arr)
pred_single = np.clip(pipe_full.predict(X_test), 0, 1)
print(f"  Train MAE (full): {mean_absolute_error(y_arr, pipe_full.predict(X)):.8f}")

# --- 8-2. 방법 2: K-Fold Averaging ---
print(f"\n[방법 2] {N_SPLITS}-Fold Averaging")

kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
oof_preds  = np.zeros(len(y_arr))
fold_preds = np.zeros((len(X_test), N_SPLITS))

for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
    X_tr, X_val = X[tr_idx], X[val_idx]
    y_tr, y_val = y_arr[tr_idx], y_arr[val_idx]

    pipe = build_svr_pipeline(best_params, len(y_tr))
    pipe.fit(X_tr, y_tr)

    oof_preds[val_idx]  = pipe.predict(X_val)
    fold_preds[:, fold] = pipe.predict(X_test)

    fold_mae = mean_absolute_error(y_val, oof_preds[val_idx])
    print(f"  Fold {fold+1:2d} MAE: {fold_mae:.5f}")

pred_kfold = fold_preds.mean(axis=1)
oof_mae = mean_absolute_error(y_arr, oof_preds)
print(f"  OOF MAE: {oof_mae:.5f}")

# --- 8-3. 방법 3: Multi-Seed Averaging ---
print(f"\n[방법 3] Multi-Seed Averaging")
seeds = [42, 123, 456, 789, 2024, 2025, 2026, 777, 1234, 5678]
seed_preds = []

for seed in seeds:
    kf_s = KFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    fold_p = []
    for tr_idx, val_idx in kf_s.split(X):
        pipe = build_svr_pipeline(best_params, len(y_arr[tr_idx]))
        pipe.fit(X[tr_idx], y_arr[tr_idx])
        fold_p.append(pipe.predict(X_test))
    seed_preds.append(np.mean(fold_p, axis=0))

pred_multiseed = np.mean(seed_preds, axis=0)
print(f"  {len(seeds)} seeds × {N_SPLITS} folds = "
      f"{len(seeds) * N_SPLITS}개 모델 평균 완료")

# ============================================================
# 9. 블렌딩 및 제출 파일 저장
# ============================================================
print("\n[블렌딩]")

# Full train은 오버핏 경향 → 비중 낮게 설정
pred_blend = (
    0.2 * pred_single +      # Full train (공격적)
    0.3 * pred_kfold +       # K-Fold 평균 (안정적)
    0.5 * pred_multiseed     # Multi-Seed 평균 (가장 안정적)
)
pred_blend = np.clip(pred_blend, 0, 1)

print(f"  블렌딩 예측 통계: mean={pred_blend.mean():.4f}, "
      f"std={pred_blend.std():.4f}")

# 메인 제출 파일
submission = submit.copy()
submission['stress_score'] = pred_blend
submission.to_csv(f'{DATA_PATH}submission.csv', index=False)
print(f"\n  ✅ 제출 파일 저장: {DATA_PATH}submission.csv")

# ============================================================
# 10. (보너스) epsilon=0 버전
# ============================================================
print(f"\n[보너스] epsilon=0 고정 + C/gamma만 최적화")

def objective_eps0(params):
    """epsilon=0 고정 SVR 목적함수."""
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    fold_maes = []
    for tr_idx, val_idx in kf.split(X):
        pipe = make_pipeline(
            RobustScaler(),
            TransformedTargetRegressor(
                regressor=SVR(
                    kernel="rbf",
                    C=params["C"],
                    gamma=params["gamma"],
                    epsilon=0.0,
                    shrinking=True,
                    cache_size=500
                ),
                transformer=QuantileTransformer(
                    output_distribution="normal",
                    n_quantiles=min(1000, len(y_arr[tr_idx]))
                )
            )
        )
        pipe.fit(X[tr_idx], y_arr[tr_idx])
        pred = pipe.predict(X[val_idx])
        fold_maes.append(mean_absolute_error(y_arr[val_idx], pred))
    return {"loss": float(np.mean(fold_maes)), "status": STATUS_OK}


space_eps0 = {
    "C":     hp.loguniform("C",     np.log(0.5),  np.log(50)),
    "gamma": hp.loguniform("gamma", np.log(0.05), np.log(5)),
}

print(f"  epsilon=0 Hyperopt 탐색 ({HYPEROPT_EVALS}회)...")
trials2 = Trials()
best2 = fmin(
    fn=objective_eps0,
    space=space_eps0,
    algo=tpe.suggest,
    max_evals=HYPEROPT_EVALS,
    trials=trials2,
    rstate=np.random.default_rng(RANDOM_STATE)
)

best2_params = {"C": float(best2["C"]), "gamma": float(best2["gamma"])}
best2_mae = float(min(r["loss"] for r in trials2.results))
print(f"  eps=0 Best CV MAE: {best2_mae:.5f}")
print(f"  C={best2_params['C']:.4f}, gamma={best2_params['gamma']:.4f}")

# eps=0 Multi-Seed Averaging
seed_preds_eps0 = []
for seed in seeds:
    kf_s = KFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    fp = []
    for tr_idx, val_idx in kf_s.split(X):
        p = make_pipeline(
            RobustScaler(),
            TransformedTargetRegressor(
                regressor=SVR(
                    kernel="rbf",
                    C=best2_params["C"],
                    gamma=best2_params["gamma"],
                    epsilon=0.0,
                    shrinking=True,
                    cache_size=500
                ),
                transformer=QuantileTransformer(
                    output_distribution="normal",
                    n_quantiles=min(1000, len(y_arr[tr_idx]))
                )
            )
        )
        p.fit(X[tr_idx], y_arr[tr_idx])
        fp.append(p.predict(X_test))
    seed_preds_eps0.append(np.mean(fp, axis=0))

pred_eps0_ms = np.clip(np.mean(seed_preds_eps0, axis=0), 0, 1)

# eps=0 제출 파일
sub_eps0 = submit.copy()
sub_eps0['stress_score'] = pred_eps0_ms
sub_eps0.to_csv(f'{DATA_PATH}submission_eps0.csv', index=False)
print(f"  ✅ {DATA_PATH}submission_eps0.csv 저장 완료")

# ============================================================
# 11. 최종 결과 요약
# ============================================================
print(f"\n{'=' * 65}")
print(f"  최종 결과 요약")
print(f"{'=' * 65}")
print(f"  Hyperopt Best CV MAE : {best_mae:.5f}")
print(f"  K-Fold OOF MAE       : {oof_mae:.5f}")
print(f"  eps=0 Best CV MAE    : {best2_mae:.5f}")
print(f"  최적 파라미터        : C={best_params['C']:.4f}, "
      f"gamma={best_params['gamma']:.4f}, "
      f"eps={best_params['epsilon']:.6f}")
print(f"  앙상블 구성          : 0.2×Full + 0.3×KFold + 0.5×MultiSeed")
print(f"  출력 파일:")
print(f"    1순위: {DATA_PATH}submission.csv       (블렌딩, 추천)")
print(f"    2순위: {DATA_PATH}submission_eps0.csv   (eps=0 MultiSeed)")
print(f"{'=' * 65}")